# Pruebas de significancia, ablación y tiempos de cómputo

Este cuaderno ejecuta `src/pruebas.py`: métricas recomendadas en la literatura, DeLong, bootstrap por conglomerados (municipios), McNemar, Friedman + Nemenyi con las semanas como bloques, la ablación del modelo original y los tiempos de cómputo.

In [1]:
import os  # Variables de entorno
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path.insert(0, "..")  # Agrega la raíz del proyecto para importar src y dashboard
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones que no afectan los resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
from IPython.display import HTML, display  # Incrustar figuras interactivas
from dashboard import figuras as fg  # Mismas figuras del dashboard
from src import config as cf  # Configuración del proyecto
pd.set_option("display.width", 160)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles


def mostrar(fig):  # Muestra una figura del dashboard como HTML interactivo
    """Aplica el fondo oscuro de las tarjetas y la incrusta con Plotly cargado desde internet."""
    fig.update_layout(paper_bgcolor=fg.SUPERFICIE, plot_bgcolor=fg.SUPERFICIE)  # Fondo oscuro
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))  # Figura interactiva
import joblib  # Carga de resultados
from src import pruebas as pb  # Pruebas
pb.main()  # Ejecuta todo (varios minutos)
P = joblib.load(cf.RUTA_PRUEBAS)  # Resultados

Métricas listas


HetDS vs LogisticRegression: cluster AUC p=0.001


HetDS vs RandomForest: cluster AUC p=0.001


HetDS vs KNN: cluster AUC p=0.001


HetDS vs GradientBoosting: cluster AUC p=0.001


  File "C:\DanielSR\Projects\proyecto_dengue\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\user-pc\.conda\envs\ml_venv\lib\subprocess.py", line 505, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\user-pc\.conda\envs\ml_venv\lib\subprocess.py", line 951, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\user-pc\.conda\envs\ml_venv\lib\subprocess.py", line 1436, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


Ablación lista


HetDS vs LogisticRegression   dif AUC=+0.0162  DeLong p=1.7e-85  cluster p=0.004  dif AP=+0.0198 p=0.004
HetDS vs RandomForest         dif AUC=+0.0041  DeLong p=1.2e-27  cluster p=0.004  dif AP=+0.0062 p=0.004
HetDS vs KNN                  dif AUC=+0.0162  DeLong p=2.1e-115  cluster p=0.004  dif AP=+0.0258 p=0.004
HetDS vs GradientBoosting     dif AUC=+0.0040  DeLong p=4.2e-25  cluster p=0.004  dif AP=+0.0062 p=0.004
Friedman: 2.9566430933552024e-58 {'LogisticRegression': 4.36, 'RandomForest': 2.52, 'KNN': 4.39, 'GradientBoosting': 2.24, 'HetDS': 1.49}
GradientBoosting (variables base) {'auc_roc': 0.8977, 'auc_pr': 0.8226, 'dif_gb': 0.0, 'ic': (0.0, 0.0), 'p': 1.0}
GradientBoosting + espaciales {'auc_roc': 0.9005, 'auc_pr': 0.8267, 'dif_gb': 0.0028, 'ic': (0.0017771707645833606, 0.0037068891060057616), 'p': 0.002}
HetDS sin espaciales (en línea) {'auc_roc': 0.8994, 'auc_pr': 0.8258, 'dif_gb': 0.0016, 'ic': (0.0009316707509727489, 0.002391398851496701), 'p': 0.002}
HetDS completo (en lí

## Métricas de comparación

In [2]:
m = pd.DataFrame(P["metricas"]).T.drop(columns=["ic_auc_roc", "ic_auc_pr"]).rename(index=fg.ETIQUETAS_MODELOS)  # Métricas
m.round(4).sort_values("auc_pr", ascending=False)  # Tabla

,auc_roc,auc_pr,mcc,bal_acc,brier,logloss,prec_k,rec_k
HetDS en línea (original),0.901679,0.828781,0.610334,0.819301,0.111701,0.355843,0.988298,0.163054
RandomForestClassifier,0.897585,0.822605,0.606776,0.814004,0.114404,0.365866,0.985106,0.162527
GradientBoostingClassifier,0.897722,0.822571,0.607333,0.812007,0.11419,0.365359,0.990426,0.163405
LogisticRegression,0.885447,0.808961,0.593176,0.803075,0.121782,0.391955,0.985638,0.162615
KNeighborsClassifier,0.885445,0.803027,0.583035,0.806098,0.123537,0.419029,0.973404,0.160597


## Pruebas pareadas (HetDS frente a cada modelo, p ajustados por Holm)

In [3]:
filas = []  # Una fila por modelo
for n, d in P["pruebas"].items():  # Resultados
    filas.append({"HetDS vs.": fg.ETIQUETAS_MODELOS[n], "ΔAUC-ROC": d["delong"]["dif"], "p DeLong": d["delong"]["p_holm"],  # DeLong
                  "IC 95 % bootstrap": tuple(round(x, 4) for x in d["cluster_roc"]["ic"]), "p bootstrap (ROC)": d["cluster_roc"]["p_holm"],  # Bootstrap ROC
                  "ΔAUC-PR": d["cluster_pr"]["dif"], "p bootstrap (PR)": d["cluster_pr"]["p_holm"], "McNemar b/c": f"{d['mcnemar']['b']}/{d['mcnemar']['c']}",
                  "p McNemar": d["mcnemar"]["p_holm"]})
pd.DataFrame(filas).set_index("HetDS vs.")  # Tabla

,ΔAUC-ROC,p DeLong,IC 95 % bootstrap,p bootstrap (ROC),ΔAUC-PR,p bootstrap (PR),McNemar b/c,p McNemar
HetDS vs.,,,,,,,,
LogisticRegression,0.016232,1.704322e-85,"(0.0138, 0.0189)",0.004,0.019820,0.004,1289/1286,9.685610e-01
RandomForestClassifier,0.004094,1.151971e-27,"(0.0031, 0.0051)",0.004,0.006176,0.004,694/779,5.724341e-02
KNeighborsClassifier,0.016235,2.147918e-115,"(0.0143, 0.0184)",0.004,0.025754,0.004,1394/884,5.967306e-26
GradientBoostingClassifier,0.003958,4.169657e-25,"(0.0029, 0.005)",0.004,0.006210,0.004,719/893,4.922681e-05


In [4]:
mostrar(fg.fig_nemenyi(P["friedman"]["auc_roc"]))  # Friedman + Nemenyi semanal
print({k: v for k, v in P["friedman"]["auc_roc"].items() if k != "rangos"})  # Estadístico, p y DC

{'chi2': 274.7919999999999, 'p': 2.9566430933552024e-58, 'cd': 0.6099993442619427, 'n_semanas': 100, 'gana_hetds': 0.66}


## Ablación: ¿de dónde viene la ganancia?

In [5]:
display(pd.DataFrame(P["ablacion"]).T)  # Variantes
mostrar(fg.fig_ablacion(P["ablacion"]))  # Figura

,auc_roc,auc_pr,dif_gb,ic,p
GradientBoosting (variables base),0.897722,0.822571,0.0,"(0.0, 0.0)",1.0
GradientBoosting + espaciales,0.900498,0.826714,0.002777,"(0.0017771707645833606, 0.0037068891060057616)",0.002
HetDS sin espaciales (en línea),0.89936,0.825801,0.001639,"(0.0009316707509727489, 0.002391398851496701)",0.002
HetDS completo (en línea),0.901679,0.828781,0.003958,"(0.0029120504814233727, 0.005006018151283292)",0.002
HetDS estático (pesos iguales),0.90115,NaN,NaN,NaN,NaN


## Tiempos de cómputo

In [6]:
display(pd.DataFrame(P["tiempos"]).T.rename(index=fg.ETIQUETAS_MODELOS).round(2))  # Tabla
mostrar(fg.fig_tiempos(P["tiempos"]))  # Búsqueda y un ajuste
mostrar(fg.fig_prediccion(P["tiempos"]))  # Latencia de predicción

,grid_s,fit_s,n_fit,pred_ms_1000
LogisticRegression,13.1,1.03,134232.0,0.67
RandomForestClassifier,700.5,173.37,134232.0,27.17
KNeighborsClassifier,19.4,0.10,30000.0,31.28
GradientBoostingClassifier,1070.2,283.74,134232.0,2.29
HetDS en línea (original),79.3,77.66,134232.0,21.03
